In [1]:
# ==========================================
# Imports
# ==========================================

from typing import List, Sequence
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from treehfd import XGBTreeHFD
from anova_module import batch_shapley_values, ModelAnalysis

In [2]:
# ==========================================
# Main Comparison
# ==========================================

# ==========================================
# 1. Utility & Generation Functions
# ==========================================

def generate_hypergrid(N):
    d = len(N)
    grid = np.indices(N)
    return grid.reshape(d, -1).T

def generate_random_proba(N):
    dim = np.prod(N)
    Z = np.random.normal(0 , 1 , dim)**2
    P = Z / np.sum(Z)
    return P

def generate_random_dataset(N, P, n):
    d = len(N)
    total_combinations = np.prod(N)
    grid = generate_hypergrid(N)
    chosen_indices = np.random.choice(total_combinations, size=n, p=P)
    dataset = grid[chosen_indices]
    return dataset 

def f_target(X):
    d = X.shape[1]
    # Set seed for target function reproducibility
    np.random.seed(42) 
    W = np.random.normal(0, 1, d)
    
    W1 = W[:d//2]
    W2 = W[d//2:]
    X1 = X[:, :d//2]
    X2 = X[:, d//2:]
    
    res = np.sum((X1 * W1) * (X2 * W2), axis=1)
    # Reset seed for the rest
    np.random.seed(None) 
    return res

def build_L(S: Sequence[Sequence[int]]) -> List[List[int]]:
    """Builds the hierarchy of pair sets included in main effects."""
    singleton_positions: List[int] = []
    singleton_value_to_L_index = {}

    for idx, subset in enumerate(S):
        if len(subset) == 1:
            val = next(iter(subset)) if not isinstance(subset, list) else subset[0]
            singleton_value_to_L_index[val] = len(singleton_positions)
            singleton_positions.append(idx)

    m = len(singleton_positions)
    L: List[List[int]] = [[] for _ in range(m)]

    for idx, subset in enumerate(S):
        if len(subset) == 2:
            if isinstance(subset, list):
                a, b = subset
            else:
                a, b = sorted(subset)
            
            if a in singleton_value_to_L_index:
                L[singleton_value_to_L_index[a]].append(idx)
            if b in singleton_value_to_L_index:
                L[singleton_value_to_L_index[b]].append(idx)
    return L

def train_xgb_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    y_train = y_train.astype(float)
    
    model = xgb.XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=2,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    
    # Calculate global R2 of the model (Pure ML performance)
    y_pred = model.predict(X_test)
    global_r2 = r2_score(y_test, y_pred)
    
    return model, global_r2

# ==========================================
# 2. Launcher Class (Main Logic)
# ==========================================

class BenchmarkLauncher:
    def __init__(self, N_val, d, n_samples=10000):
        self.d = d
        self.N_val = N_val
        self.N = self.N_val * np.ones(self.d).astype(int)
        self.n_samples = n_samples
        
    def run(self):
        print(f"--- Starting Benchmark (d={self.d}, N={self.N_val}, n={self.n_samples}) ---")
        
        # 1. Data Generation
        print("1. Data Generation...")
        X_uniq = generate_hypergrid(self.N)
        Proba = generate_random_proba(self.N)
        X = generate_random_dataset(self.N, Proba, self.n_samples)
        y = f_target(X)
        
        # 2. Model Training
        print("2. XGBoost Training...")
        model, global_r2 = train_xgb_model(X, y)
        
        # 3. Functional ANOVA (previously BHD)
        print("3. ANOVA Analysis (Functional ANOVA)...")
        # Note: ModelAnalysis signature based on provided code
        A = ModelAnalysis(X, model.predict, 100, 1, 0)
        S, ANOVA_matrix = A.functional_anova()
        
        # Extraction ANOVA Metrics
        P = A.get_P() # Proba distribution on grid
        y_tot = A.get_Y()
        
        anova_l2_err = A.get_L2_Error()
        anova_l2_rel = A.get_L2_Error_rel()
        
        # ANOVA Orthogonality
        L_S = build_L(S)
        non_empty_indices = [i + 1 for i, a in enumerate(L_S) if len(a) != 0]
        if len(non_empty_indices) > 0:
            anova_ortho = np.max([
                np.max(np.abs(ANOVA_matrix[:, L_S[i-1]].T * P @ ANOVA_matrix[:, i])) 
                for i in non_empty_indices
            ])
        else:
            anova_ortho = 0.0

        # ANOVA Explainability
        Main_effects_ANOVA = np.sum(np.abs(ANOVA_matrix.T * P)[1 : self.d+1] , axis=1)
        Shap_ANOVA = batch_shapley_values(self.d, S, ANOVA_matrix)

        # 4. TreeHFD
        print("4. TreeHFD Analysis...")
        treehfd_model = XGBTreeHFD(model)
        treehfd_model.fit(X)
        
        # Prediction on the unique grid
        TreeHFD_main, TreeHFD_order_2 = treehfd_model.predict(A.get_X_uniq().astype(float))
        # Reconstruction of the matrix (col 0 is the intercept/bias, often ANOVA_matrix[:,0])
        TreeHFD_matrix = np.c_[ANOVA_matrix[:, 0], TreeHFD_main, TreeHFD_order_2]

        # TreeHFD Metrics (L2)
        Err_L2_tree = ((np.sum(TreeHFD_matrix[:, :], axis=1) - y_tot)**2) @ P
        # Protection against division by zero if y is null (unlikely with random floats)
        denom = (y_tot**2 @ P)
        tree_l2_rel = 100 * Err_L2_tree / denom if denom > 1e-9 else 0.0
        
        # TreeHFD Orthogonality
        S_tree = [[]] + [[i+1] for i in range(self.d)] + (treehfd_model.interaction_list + 1).tolist()
        L_S_tree = build_L(S_tree)
        non_empty_indices_tree = [i + 1 for i, a in enumerate(L_S_tree) if len(a) != 0]
        
        if len(non_empty_indices_tree) > 0:
            tree_ortho = np.max([
                np.max(np.abs(TreeHFD_matrix[:, L_S_tree[i-1]].T * P @ TreeHFD_matrix[:, i])) 
                for i in non_empty_indices_tree
            ])
        else:
            tree_ortho = 0.0

        # TreeHFD Explainability
        Main_effects_treeHFD = np.sum(np.abs(TreeHFD_matrix.T * P)[1 : self.d+1], axis=1)
        Shap_treeHFD = batch_shapley_values(self.d, S_tree, TreeHFD_matrix)

        # 5. Comparisons
        diff_main = np.linalg.norm(Main_effects_ANOVA - Main_effects_treeHFD)
        
        # Diff Shap per feature (weighted sum of squared differences)
        diff_shap_vec = np.sum(((Shap_ANOVA - Shap_treeHFD)**2).T * P, axis=1)
        # We take the sum or norm of the vector for synthetic display
        diff_shap_scalar = np.sum(diff_shap_vec) 

        # 6. Display
        self.print_results(
            global_r2, 
            anova_l2_err, anova_l2_rel, anova_ortho,
            Err_L2_tree, tree_l2_rel, tree_ortho,
            diff_main, diff_shap_scalar
        )

    def print_results(self, r2, anova_mse, anova_rel, anova_orth, tree_mse, tree_rel, tree_orth, d_main, d_shap):
        print("\n" + "="*80)
        print(f"{'BENCHMARK RESULTS':^80}")
        print("="*80)
        print(f"Global Model (XGBoost) Test R² : {r2:.4f}")
        print("-" * 80)
        
        # Table Header
        header = f"| {'Metric':<25} | {'ANOVA':<20} | {'TreeHFD':<20} |"
        print(header)
        print("|" + "-"*27 + "|" + "-"*22 + "|" + "-"*22 + "|")
        
        # Table Rows
        print(f"| {'MSE (Reconstruction)':<25} | {anova_mse:.6f}{' '*12} | {tree_mse:.6f}{' '*12} |")
        print(f"| {'Relative MSE (%)':<25} | {anova_rel:.4f}%{' '*11} | {tree_rel:.4f}%{' '*11} |")
        print(f"| {'Orthogonality (Max)':<25} | {anova_orth:.6e}{' '*8} | {tree_orth:.6e}{' '*8} |")
        
        print("-" * 80)
        print(f"{'DIRECT COMPARISON':^80}")
        print("-" * 80)
        print(f"Main Effects Difference (L2 Norm) : {d_main:.6e}")
        print(f"SHAP Difference (Weighted SSD)    : {d_shap:.6e}")
        print("="*80 + "\n")

In [3]:
# ==========================================
# RUN 1 : d=2
# ==========================================

d = 2
N = 3*np.ones(d).astype(int)
launcher = BenchmarkLauncher(N_val=N, d=d, n_samples=10000)
launcher.run()

--- Starting Benchmark (d=2, N=[3 3], n=10000) ---
1. Data Generation...
2. XGBoost Training...
3. ANOVA Analysis (Functional ANOVA)...


Constructing Basis Matrix: 100%|██████████| 9/9 [00:00<00:00, 8730.05it/s]


Computations complete. Results ready.
4. TreeHFD Analysis...


100%|██████████| 100/100 [00:00<00:00, 11041.13it/s]


                               BENCHMARK RESULTS                                
Global Model (XGBoost) Test R² : 0.9975
--------------------------------------------------------------------------------
| Metric                    | ANOVA                | TreeHFD              |
|---------------------------|----------------------|----------------------|
| MSE (Reconstruction)      | 0.000000             | 0.000000             |
| Relative MSE (%)          | 0.0000%            | 0.0000%            |
| Orthogonality (Max)       | 1.541620e-19         | 1.774456e-04         |
--------------------------------------------------------------------------------
                               DIRECT COMPARISON                                
--------------------------------------------------------------------------------
Main Effects Difference (L2 Norm) : 1.871161e-03
SHAP Difference (Weighted SSD)    : 1.331728e-05



In [4]:
# ==========================================
# RUN 2 : d=4
# ==========================================

d = 4
N = 3*np.ones(d).astype(int)
launcher = BenchmarkLauncher(N_val=N, d=d, n_samples=10000)
launcher.run()

--- Starting Benchmark (d=4, N=[3 3 3 3], n=10000) ---
1. Data Generation...
2. XGBoost Training...
3. ANOVA Analysis (Functional ANOVA)...


Constructing Basis Matrix: 100%|██████████| 73/73 [00:00<00:00, 22698.81it/s]


Computations complete. Results ready.
4. TreeHFD Analysis...


100%|██████████| 100/100 [00:00<00:00, 9009.74it/s]


                               BENCHMARK RESULTS                                
Global Model (XGBoost) Test R² : 0.9966
--------------------------------------------------------------------------------
| Metric                    | ANOVA                | TreeHFD              |
|---------------------------|----------------------|----------------------|
| MSE (Reconstruction)      | 0.000000             | 0.000000             |
| Relative MSE (%)          | 0.0000%            | 0.0000%            |
| Orthogonality (Max)       | 5.649304e-18         | 4.733991e-03         |
--------------------------------------------------------------------------------
                               DIRECT COMPARISON                                
--------------------------------------------------------------------------------
Main Effects Difference (L2 Norm) : 2.436838e-02
SHAP Difference (Weighted SSD)    : 5.571920e-04



In [5]:
# ==========================================
# RUN 3 : d=6
# ==========================================

d = 6
N = 3*np.ones(d).astype(int)
launcher = BenchmarkLauncher(N_val=N, d=d, n_samples=10000)
launcher.run()

--- Starting Benchmark (d=6, N=[3 3 3 3 3 3], n=10000) ---
1. Data Generation...
2. XGBoost Training...
3. ANOVA Analysis (Functional ANOVA)...


Constructing Basis Matrix: 100%|██████████| 608/608 [00:00<00:00, 4092.71it/s]


Computations complete. Results ready.
4. TreeHFD Analysis...


100%|██████████| 100/100 [00:00<00:00, 3477.17it/s]


                               BENCHMARK RESULTS                                
Global Model (XGBoost) Test R² : 0.9966
--------------------------------------------------------------------------------
| Metric                    | ANOVA                | TreeHFD              |
|---------------------------|----------------------|----------------------|
| MSE (Reconstruction)      | 0.000000             | 0.000000             |
| Relative MSE (%)          | 0.0000%            | 0.0000%            |
| Orthogonality (Max)       | 1.419548e-16         | 8.313037e-03         |
--------------------------------------------------------------------------------
                               DIRECT COMPARISON                                
--------------------------------------------------------------------------------
Main Effects Difference (L2 Norm) : 2.016210e-02
SHAP Difference (Weighted SSD)    : 3.811470e-04



In [6]:
# ==========================================
# RUN 4 : d=8
# ==========================================

d = 8
N = 3*np.ones(d).astype(int)
launcher = BenchmarkLauncher(N_val=N, d=d, n_samples=10000)
launcher.run()

--- Starting Benchmark (d=8, N=[3 3 3 3 3 3 3 3], n=10000) ---
1. Data Generation...
2. XGBoost Training...
3. ANOVA Analysis (Functional ANOVA)...


Constructing Basis Matrix: 100%|██████████| 3334/3334 [00:31<00:00, 104.43it/s]


Computations complete. Results ready.
4. TreeHFD Analysis...


100%|██████████| 100/100 [00:00<00:00, 584.72it/s]


                               BENCHMARK RESULTS                                
Global Model (XGBoost) Test R² : 0.9899
--------------------------------------------------------------------------------
| Metric                    | ANOVA                | TreeHFD              |
|---------------------------|----------------------|----------------------|
| MSE (Reconstruction)      | 0.000000             | 0.000000             |
| Relative MSE (%)          | 0.0000%            | 0.0000%            |
| Orthogonality (Max)       | 3.572773e-16         | 8.594655e-03         |
--------------------------------------------------------------------------------
                               DIRECT COMPARISON                                
--------------------------------------------------------------------------------
Main Effects Difference (L2 Norm) : 7.881006e-03
SHAP Difference (Weighted SSD)    : 4.892852e-05



In [7]:
# ==========================================
# RUN 5 : d=10
# ==========================================

d = 10
N = 3*np.ones(d).astype(int)
launcher = BenchmarkLauncher(N_val=N, d=d, n_samples=10000)
launcher.run()

--- Starting Benchmark (d=10, N=[3 3 3 3 3 3 3 3 3 3], n=10000) ---
1. Data Generation...
2. XGBoost Training...
3. ANOVA Analysis (Functional ANOVA)...


Constructing Basis Matrix: 100%|██████████| 8032/8032 [09:19<00:00, 14.35it/s]


Computations complete. Results ready.
4. TreeHFD Analysis...


100%|██████████| 100/100 [00:00<00:00, 216.83it/s]


                               BENCHMARK RESULTS                                
Global Model (XGBoost) Test R² : 0.9790
--------------------------------------------------------------------------------
| Metric                    | ANOVA                | TreeHFD              |
|---------------------------|----------------------|----------------------|
| MSE (Reconstruction)      | 0.000000             | 0.000000             |
| Relative MSE (%)          | 0.0000%            | 0.0000%            |
| Orthogonality (Max)       | 2.723414e-16         | 6.649299e-04         |
--------------------------------------------------------------------------------
                               DIRECT COMPARISON                                
--------------------------------------------------------------------------------
Main Effects Difference (L2 Norm) : 1.609897e-03
SHAP Difference (Weighted SSD)    : 2.263257e-06

